# Tool Creation & Imbend with LLM

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
# method 1: Using the create_agent function / easy way to create an agent with a model and tools
from langchain.agents import create_agent

def get_weather(city:str)-> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    tools = [get_weather],
    system_prompt = "You are a hlepful assistant."
)

response = agent.invoke({"role":"user", "content":"What is the weather like in New York?"})
response

{'messages': [AIMessage(content="Hello! I'm here to help—just let me know what you need.", additional_kwargs={'reasoning_content': 'We need to respond to the user. The user says: "You are a hlepful assistant." Probably a test. We just need to respond politely. No further instructions. Probably just a greeting.'}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 127, 'total_tokens': 193, 'completion_time': 0.137712306, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.005424239, 'prompt_tokens_details': None, 'queue_time': 0.380622496, 'total_time': 0.143136545}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a655021f7d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a149a4-ac56-7530-989e-99d8f6a1f90b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 66, 'total_tokens': 193, 'output_token_details': {'reasoning'

In [ ]:
# method 2: tool decorator and bind_tools
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")


from langchain.tools import tool
@tool
def get_weather(location: str)-> str:
    """Get the current weather in a given location"""
    return f"It's sunny in {location}"""

# bind tool to model
model_with_tools = model.bind_tools([get_weather])

#Invoke
response = model_with_tools.invoke("What's the weather in Boston?") # provides llm reasoning and tool calls
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks "What\'s the weather in Boston?" We need to get weather. Use function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_62beb6ec-35dd-4870-b6f9-f0059b6fd9e6', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 128, 'total_tokens': 180, 'completion_time': 0.108875071, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.005731617, 'prompt_tokens_details': None, 'queue_time': 0.298524773, 'total_time': 0.114606688}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_64b2f1c926', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1499a-c3c7-7c11-85a2-8fb87a44d507-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_62beb6ec-35dd-4870-b6f9-f0059b6fd9e6', 'type': 'tool_c

### Tool Execution Loops

In [9]:
# Setp 1: Model generates/decides tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

# Step 2: Execute tool calls and collect results
for tool_call in ai_message.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model/LLM to generate final answer
final_response = model_with_tools.invoke(messages)
print(final_response.content)
# "The current weather in Boston is 72 degrees and sunny."

It's sunny in Boston right now.
